In [6]:
import torch
import torch.nn.functional as F
import math

In [8]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Calculates the Scaled Dot-Product Attention.

    Args:
        query (torch.Tensor): Query tensor; shape (batch_size, ..., seq_len_q, d_k)
        (torch.Tensor): Key tensor; shape (batch_size, ..., seq_len_k, d_k)
        value (torch.Tensor): Value tensor; shape (batch_size, ..., seq_len_v, d_v)
                                Note: seq_len_k and seq_len_v must be the same.
        mask (torch.Tensor, optional): Mask tensor; shape must be broadcastable
                                       to (batch_size, ..., seq_len_q, seq_len_k).
                                       Defaults to None.

    Returns:
        torch.Tensor: Output tensor; shape (batch_size, ..., seq_len_q, d_v)
        torch.Tensor: Attention weights; shape (batch_size, ..., seq_len_q, seq_len_k)
    """
    # Get the dimension of the vectors
    d_k = query.size(-1)

    # 1. Calculate dot products: Q * K^T
    # Result shape: (batch_size, ..., seq_len_q, seq_len_k)

    attention_scores = torch.matmul(query, key.transpose(-2, -1))

    # 2. Scale the scores
    attention_scores = attention_scores / math.sqrt(d_k)

    # 3. Apply the mask (if provided)
    # The mask indicates positions to ignore (e.g., padding).
    # We add a large negative number (-1e9) to these positions before softmax.
    if mask is not None:
        # Ensure mask has compatible shape
        attention_scores = attention_scores.masked_fill(mask == 0, -1e9)

    # 4. Apply softmax to get attention weights
    # Softmax is applied on the last dimension (seq_len_k)
    # Result shape: (batch_size, ..., seq_len_q, seq_len_k)
    attention_weights = F.softmax(attention_scores, dim=-1)

    # 5. Multiply weights by Value vectors V
    # Result shape: (batch_size, ..., seq_len_q, d_v)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

In [17]:
batch_size = 10
seq_len = 4
d_k = 8 # Dimension of Key/Query
d_v = 8 # Dimension of Value

# Create random Query, Value tensors
# In a real model, these would come from input embeddings projected by linear layers
query = torch.randn(batch_size, seq_len, d_k)
key = torch.randn(batch_size, seq_len, d_k)
value = torch.randn(batch_size, seq_len, d_v)

# Calculate attention
output, attention_weights = scaled_dot_product_attention(query, key, value)

print("Input Query Shape:", query.shape)
print("Input Shape:", key.shape)
print("Input Value Shape:", value.shape)
print("\nOutput Shape:", output.shape)
print("Attention Weights Shape:", attention_weights.shape)
print("\nSample Attention Weights (first batch element):\n", attention_weights[9])

Input Query Shape: torch.Size([10, 4, 8])
Input Shape: torch.Size([10, 4, 8])
Input Value Shape: torch.Size([10, 4, 8])

Output Shape: torch.Size([10, 4, 8])
Attention Weights Shape: torch.Size([10, 4, 4])

Sample Attention Weights (first batch element):
 tensor([[0.5940, 0.2066, 0.0548, 0.1446],
        [0.0614, 0.0933, 0.1945, 0.6508],
        [0.1808, 0.2264, 0.3579, 0.2349],
        [0.0418, 0.4285, 0.1185, 0.4111]])


In [18]:
attention_weights

tensor([[[0.2366, 0.1708, 0.3234, 0.2692],
         [0.5707, 0.2172, 0.0670, 0.1452],
         [0.5172, 0.2144, 0.0376, 0.2308],
         [0.7983, 0.0210, 0.1738, 0.0069]],

        [[0.5990, 0.1882, 0.0526, 0.1603],
         [0.2162, 0.4120, 0.1508, 0.2210],
         [0.6325, 0.0930, 0.0886, 0.1860],
         [0.0579, 0.1694, 0.1858, 0.5869]],

        [[0.1652, 0.0341, 0.5874, 0.2133],
         [0.3473, 0.1084, 0.0540, 0.4903],
         [0.2371, 0.1457, 0.3469, 0.2703],
         [0.2174, 0.0630, 0.3185, 0.4010]],

        [[0.2790, 0.5523, 0.0604, 0.1084],
         [0.0744, 0.5627, 0.0605, 0.3024],
         [0.0857, 0.7737, 0.0645, 0.0760],
         [0.2133, 0.1689, 0.3557, 0.2622]],

        [[0.2786, 0.3460, 0.3112, 0.0642],
         [0.3646, 0.1440, 0.4450, 0.0464],
         [0.0450, 0.1998, 0.4900, 0.2652],
         [0.1627, 0.2465, 0.1722, 0.4186]],

        [[0.2491, 0.0705, 0.1266, 0.5539],
         [0.2168, 0.1977, 0.3949, 0.1907],
         [0.1524, 0.4640, 0.2005, 0.1831],
 

## Encoding layer

In [1]:
import torch
import torch.nn as nn

class PositionWiseFF(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):

        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)

        return x

In [1]:
!pip install transformers[torch]

  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/11.2 MB 5.2 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.2 MB 13.0 MB/s eta 0:00:01
   ----------------------------------- ----


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Note: This is code to illustrate the workflow.
# Actual implementation might vary slightly based on the task.

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch # Or tensorflow if using TF

# 1. Choose a pre-trained model checkpoint
model_name = "bert-base-uncased" # Example: BERT model

# 2. Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Load the pre-trained model (here, for sequence classification)
#    Loading 'AutoModel' would give the base Transformer without a specific head.
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 4. Prepare input text
raw_text = ["This is the first sentence.", "This is the first sentence"]
inputs = tokenizer(raw_text, padding=True, truncation=True, return_tensors="pt")
# 'inputs' now contains input_ids, attention_mask, etc. as PyTorch tensors ("pt")

# 5. Perform inference (get model outputs)
with torch.no_grad(): # Disable gradient calculation for inference
    outputs = model(**inputs)
    logits = outputs.logits # Raw scores from the classification head

# (Optional) Further processing: apply softmax, map to labels, etc.
probabilities = torch.softmax(logits, dim=-1)
predicted_classes = torch.argmax(probabilities, dim=-1)

print(f"Input IDs shape: {inputs['input_ids'].shape}")
print(f"Logits shape: {logits.shape}")
print(f"Predicted classes: {predicted_classes}")

print(outputs)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11026.41it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Input IDs shape: torch.Size([2, 8])
Logits shape: torch.Size([2, 2])
Predicted classes: tensor([0, 0])
SequenceClassifierOutput(loss=None, logits=tensor([[-0.0177, -0.0951],
        [ 0.0372, -0.1705]]), hidden_states=None, attentions=None)
